In [1]:
from transformers import AutoTokenizer
import json
from datasets import Dataset
import numpy as np
# from skmultilearn.model_selection import iterative_train_test_split
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

In [60]:
file_path1 = "data/stratified_test_data_topic.json"  # Update with your actual file path
with open(file_path1, "r") as f:
    data1 = json.load(f)
file_path2 = "data/stratified_train_data_topic.json"  # Update with your actual file path
with open(file_path2, "r") as f:
    data2 = json.load(f)

In [61]:
for d in data1 +data2:
    for l in d['labels']:
        flag = False
        if "Reinforcement" in l:
            print (d)
            flag = True
            break
    if flag:
        break
        

{'text': 'From Provider: Good afternoon Mr. Person1, A 30 day supply is being sent to your pharmacy as a courtesy. Thank you, Person2', 'labels': ['PartnershipProvider_salutation', 'PartnershipProvider_signoff', 'SharedDecisionProvider_Approval/Reinforcement']}


In [62]:
all_labels = set()
for dataset in [data1, data2]:
    for example in dataset:
        for label in example["labels"]:
            all_labels.add(label)
all_labels = sorted(all_labels)
all_labels

['CareCoordinationPatient_None',
 'CareCoordinationProvider_None',
 'PartnershipPatient_Appreciation/Gratitude',
 'PartnershipPatient_Clinical care',
 'PartnershipPatient_activeParticipation/involvement',
 'PartnershipPatient_alignment',
 'PartnershipPatient_build trust',
 'PartnershipPatient_connection',
 'PartnershipPatient_expressOpinions',
 'PartnershipPatient_salutation',
 'PartnershipPatient_signoff',
 'PartnershipPatient_statePreferences',
 'PartnershipProvider_Appreciation/Gratitude',
 'PartnershipProvider_Clinical Care',
 'PartnershipProvider_alignment',
 'PartnershipProvider_build trust',
 'PartnershipProvider_checkingUnderstanding/clarification',
 'PartnershipProvider_connection',
 'PartnershipProvider_inviteCollabration',
 'PartnershipProvider_maintainCommunication',
 'PartnershipProvider_requestsForOpinion',
 'PartnershipProvider_salutation',
 'PartnershipProvider_signoff',
 'SDOH_EconomicStability',
 'SDOH_EducationAccessAndQuality',
 'SDOH_HealthCareAccessAndQuality',
 '

In [63]:
labels_to_remove = [
    "PartnershipPatient_build trust",
    "PartnershipProvider_Appreciation/Gratitude",
    "PartnershipProvider_build trust",
    "SDOH_EducationAccessAndQuality",
    "SharedDecisionPatient_ApprovalofDecision/Reinforcement",
    "SharedDecisionProvider_Approval/Reinforcement"
]

def my_clean_label(combo):

    # if txt == "socioemotionalempathy":
    #     txt = "Socioemotional/Empathy"
    # if txt.lower() == "infoseek":
    #     txt =  "InfoSeek"
    # if txt.lower() == "infogive":
    #     txt =  "InfoGive"
    # if txt.lower() == "saddness/fear":
    #     txt =  "Sadness/fear"
    # if txt == "PositiveRemarks":
    #     txt =  "PositiveRemarks/Reward"
    # if txt.lower() == "symptoms":
    #     txt =  "Symptoms"
    # if txt.lower() == "expressconcern,unease" or txt.lower() == "expressconcern/unease":
    #     txt =  "expressconcern/unease"
    # if txt.lower() == "expressconcern,unease":
    #     txt =  "expressconcern/unease"
    # if txt.lower() == "instruction":
    #     txt =  "Instruction"
    # if txt == 'Requests for Opinion' or txt == 'acknowledgePatientExpertiseKnowledge' or txt == 'EncourageQuestions':
    #     return None
    # if txt not in standard_code and txt not in standard_subcode:
    #     return None
    # if txt == "SocioEmotionalEmpathy":
    #     txt = "SocioEmotionalBehaviour"
    # code, subcode = combo.split("_")
    remove_set = set(labels_to_remove)
    if combo in remove_set:
        return None
    return combo
def process_with_removing(data):
    processed = []
    count = 0
    for d in data:
        labels = []
        for l in d['labels']:
            if my_clean_label(l) is not None:
                labels.append(l)
            else:
                count += 1
        if len(labels) != 0:
            d["labels"] = labels
            processed.append(d)
    print (f"count is {count}")
    return processed
removed_data1 = process_with_removing(data1)
removed_data2 = process_with_removing(data2)
        

            

count is 12
count is 55


In [64]:
removed_data1[1]

{'text': 'From Patient: Hi Dr. Person1,I left two messages last week on MM/DD/YYYY and MM/DD/YYYY. Asking about stopping the eliquis for the second time on this month since I had the first procedure of diagnostic lumbar facet block.RF ABLATION.AND DR.Person 2 wants to know for my second dose is scheduled for MM/DD/YYYY he wants to know if I cud have it done that date or cud schedule it a bit further date,since I have to stop the eliquis three days prior to the procedure. Thank you, Person2. Topics: eliquis, plavix, aspirin',
 'labels': ['CareCoordinationPatient_None',
  'PartnershipPatient_Clinical care',
  'PartnershipPatient_activeParticipation/involvement',
  'PartnershipPatient_salutation',
  'PartnershipPatient_signoff',
  'SharedDecisionPatient_ExploreOptions',
  'SharedDecisionPatient_SeekingApproval']}

In [65]:
len(data1)

192

In [66]:
len(removed_data1)

192

In [67]:
len(data2)

645

In [68]:
len(removed_data2)

645

In [69]:
for d in removed_data1 + removed_data2:
    for l in d['labels']:
        flag = False
        if "Reinforcement" in l:
            print (d)
            flag = True
            break
    if flag:
        break

In [70]:
file_path1.split(".")[0] + "_remove.json"

'data/stratified_test_data_topic_remove.json'

In [71]:
with open(file_path1.split(".")[0] + "_remove.json", "w") as f:
    json.dump(removed_data1, f, indent=4)

with open(file_path2.split(".")[0] + "_remove.json", "w") as f:
    json.dump(removed_data2, f, indent=4)